In [58]:
from pathlib import Path
import numpy as np
import pandas as pd

from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

In [59]:
ocr_dir = Path("results/mol_rep_ocr/v1.1")
conversion_dir = Path("results/mol_rep_conversion/v1.1")

## OCR

In [60]:
ocr_raw_reponses = pd.DataFrame()

for file in (ocr_dir / "raw_responses").glob("*.jsonl"):
    df = pd.read_json(file, lines=True)
    ocr_raw_reponses = pd.concat([ocr_raw_reponses, df], axis=0, ignore_index=True)

In [61]:
from src.utils import compute_mol_metrics

ocr_df = ocr_raw_reponses.copy()

ocr_df[['is_gt_valid', 'is_pred_valid', 'is_em', 'is_can_smiles_match', 'is_inchikey_match', 'tanimoto_sim', '_parse_status']] = ocr_raw_reponses.apply(lambda row: compute_mol_metrics(row["completion"], row["raw_responses"], row["output_rep_type"]), axis=1, result_type="expand")

In [62]:
def aggregate_func(df):
    return pd.Series({
        "num_samples": df.shape[0],
        "gt_valid_ratio": df["is_gt_valid"].mean(),
        "pred_valid_ratio": df["is_pred_valid"].mean(),
        "em_ratio": df["is_em"].mean(),
        "can_smiles_match_ratio": df["is_can_smiles_match"].mean(),
        "inchikey_match_ratio": df["is_inchikey_match"].mean(),
        "tanimoto_sim_mean": df["tanimoto_sim"].mean(),
        "tanimoto_sim_std": df["tanimoto_sim"].std(),
    })

In [63]:
ocr_extra_test_raw_reponses = pd.DataFrame()

for file in (ocr_dir / "extra_test" / "raw_responses").glob("*.jsonl"):
    df = pd.read_json(file, lines=True)
    ocr_extra_test_raw_reponses = pd.concat([ocr_extra_test_raw_reponses, df], axis=0, ignore_index=True)

ocr_extra_test_df = ocr_extra_test_raw_reponses.copy()

ocr_extra_test_df[['is_gt_valid', 'is_pred_valid', 'is_em', 'is_can_smiles_match', 'is_inchikey_match', 'tanimoto_sim', '_parse_status']] = ocr_extra_test_raw_reponses.apply(lambda row: compute_mol_metrics(row["completion"], row["raw_responses"], row["output_rep_type"]), axis=1, result_type="expand")

### Main results

In [64]:
ocr_df.groupby("model_name").apply(aggregate_func).to_csv(ocr_dir / "metrics_overall.csv")
ocr_df.groupby("model_name").apply(aggregate_func)

,num_samples,gt_valid_ratio,pred_valid_ratio,em_ratio,can_smiles_match_ratio,inchikey_match_ratio,tanimoto_sim_mean,tanimoto_sim_std
model_name,,,,,,,,
qwen3_vl_4b_i,80.0,1.0,0.2375,0.0000,0.0125,0.0125,0.075329,0.197741
qwen3_vl_4b_i_grpo_lora_from_sft_lora_ocr_conversion,80.0,1.0,0.5625,0.2125,0.2125,0.2125,0.315708,0.403089
qwen3_vl_4b_i_sft_lora_conversion,80.0,1.0,0.6375,0.2000,0.2000,0.2000,0.337856,0.393659
qwen3_vl_4b_i_sft_lora_ocr,80.0,1.0,0.4375,0.0375,0.0375,0.0375,0.178547,0.297548
qwen3_vl_4b_i_sft_lora_ocr_conversion,80.0,1.0,0.6875,0.2625,0.2625,0.2625,0.411414,0.416413


extra testset

In [65]:
ocr_extra_test_df.groupby(["model_name"]).apply(aggregate_func).to_csv(ocr_dir / "extra_test" / "metrics_overall.csv")
ocr_extra_test_df.groupby(["model_name"]).apply(aggregate_func)

,num_samples,gt_valid_ratio,pred_valid_ratio,em_ratio,can_smiles_match_ratio,inchikey_match_ratio,tanimoto_sim_mean,tanimoto_sim_std
model_name,,,,,,,,
qwen3_vl_4b_i,80.0,1.0,0.1875,0.0000,0.0125,0.0125,0.069423,0.214454
qwen3_vl_4b_i_grpo_lora_from_sft_lora_ocr_conversion,80.0,1.0,0.5000,0.0375,0.0375,0.0375,0.164197,0.259483
qwen3_vl_4b_i_sft_lora_conversion,80.0,1.0,0.5875,0.0000,0.0000,0.0000,0.160920,0.209402
qwen3_vl_4b_i_sft_lora_ocr,80.0,1.0,0.4125,0.0375,0.0375,0.0375,0.177433,0.302280
qwen3_vl_4b_i_sft_lora_ocr_conversion,80.0,1.0,0.5500,0.0125,0.0125,0.0125,0.170351,0.228749


### Split by other dimension

In [66]:
# ocr_df.groupby("_parse_status").apply(aggregate_func).to_csv(ocr_dir / "metrics_by_parse_status.csv")
ocr_df[ocr_df._parse_status == "success"].groupby(["model_name"]).apply(aggregate_func)

,num_samples,gt_valid_ratio,pred_valid_ratio,em_ratio,can_smiles_match_ratio,inchikey_match_ratio,tanimoto_sim_mean,tanimoto_sim_std
model_name,,,,,,,,
qwen3_vl_4b_i,19.0,1.0,1.0,0.000000,0.052632,0.052632,0.317177,0.301069
qwen3_vl_4b_i_grpo_lora_from_sft_lora_ocr_conversion,45.0,1.0,1.0,0.377778,0.377778,0.377778,0.561259,0.388301
qwen3_vl_4b_i_sft_lora_conversion,51.0,1.0,1.0,0.313725,0.313725,0.313725,0.529971,0.375497
qwen3_vl_4b_i_sft_lora_ocr,35.0,1.0,1.0,0.085714,0.085714,0.085714,0.408108,0.330563
qwen3_vl_4b_i_sft_lora_ocr_conversion,55.0,1.0,1.0,0.381818,0.381818,0.381818,0.598421,0.373760


## Conversion

In [67]:
conversion_raw_reponses = pd.DataFrame()

for file in (conversion_dir / "raw_responses").glob("*.jsonl"):
    df = pd.read_json(file, lines=True)
    conversion_raw_reponses = pd.concat([conversion_raw_reponses, df], axis=0, ignore_index=True)

In [68]:
conversion_df = conversion_raw_reponses.copy()

conversion_df[['is_gt_valid', 'is_pred_valid', 'is_em', 'is_can_smiles_match', 'is_inchikey_match', 'tanimoto_sim', '_parse_status']] = conversion_raw_reponses.apply(lambda row: compute_mol_metrics(row["completion"], row["raw_responses"], row["output_rep_type"]), axis=1, result_type="expand")

In [69]:
## Extra test set
conversion_extra_test_raw_reponses = pd.DataFrame()

for file in (conversion_dir / "extra_test" / "raw_responses").glob("*.jsonl"):
    df = pd.read_json(file, lines=True)
    conversion_extra_test_raw_reponses = pd.concat([conversion_extra_test_raw_reponses, df], axis=0, ignore_index=True)

conversion_extra_test_df = conversion_extra_test_raw_reponses.copy()

conversion_extra_test_df[['is_gt_valid', 'is_pred_valid', 'is_em', 'is_can_smiles_match', 'is_inchikey_match', 'tanimoto_sim', '_parse_status']] = conversion_extra_test_df.apply(lambda row: compute_mol_metrics(row["completion"], row["raw_responses"], row["output_rep_type"]), axis=1, result_type="expand")

### Main results

In [70]:
conversion_df.groupby("model_name").apply(aggregate_func).to_csv(conversion_dir / "metrics_overall.csv")
conversion_df.groupby("model_name").apply(aggregate_func)

,num_samples,gt_valid_ratio,pred_valid_ratio,em_ratio,can_smiles_match_ratio,inchikey_match_ratio,tanimoto_sim_mean,tanimoto_sim_std
model_name,,,,,,,,
qwen3_4b_i,875.0,1.0,0.202286,0.000000,0.040000,0.040000,0.060136,0.213872
qwen3_4b_i_sft_lora_conversion,875.0,1.0,0.907429,0.750857,0.750857,0.750857,0.808812,0.362617
qwen3_vl_4b_i,875.0,1.0,0.169143,0.001143,0.070857,0.070857,0.090974,0.269518
qwen3_vl_4b_i_grpo_lora_from_sft_lora_ocr_conversion,875.0,1.0,0.806857,0.518857,0.518857,0.518857,0.612916,0.443833
qwen3_vl_4b_i_sft_lora_conversion,875.0,1.0,0.893714,0.708571,0.708571,0.708571,0.776862,0.380213
qwen3_vl_4b_i_sft_lora_ocr,875.0,1.0,0.254857,0.000000,0.050286,0.050286,0.093371,0.244099
qwen3_vl_4b_i_sft_lora_ocr_conversion,875.0,1.0,0.912000,0.731429,0.731429,0.731429,0.801808,0.362854


extra testset

In [71]:
conversion_extra_test_df.groupby(["model_name"]).apply(aggregate_func).to_csv(conversion_dir / "extra_test" / "metrics_overall.csv")
conversion_extra_test_df.groupby(["model_name"]).apply(aggregate_func)

,num_samples,gt_valid_ratio,pred_valid_ratio,em_ratio,can_smiles_match_ratio,inchikey_match_ratio,tanimoto_sim_mean,tanimoto_sim_std
model_name,,,,,,,,
qwen3_4b_i,880.0,1.0,0.203409,0.001136,0.038636,0.038636,0.054927,0.202648
qwen3_4b_i_sft_lora_conversion,880.0,1.0,0.581818,0.027273,0.030682,0.030682,0.221543,0.288056
qwen3_vl_4b_i,880.0,1.0,0.171591,0.001136,0.064773,0.064773,0.088656,0.262179
qwen3_vl_4b_i_grpo_lora_from_sft_lora_ocr_conversion,880.0,1.0,0.534091,0.021591,0.021591,0.021591,0.184784,0.270353
qwen3_vl_4b_i_sft_lora_conversion,880.0,1.0,0.623864,0.020455,0.021591,0.021591,0.234892,0.287206
qwen3_vl_4b_i_sft_lora_ocr,880.0,1.0,0.297727,0.004545,0.047727,0.047727,0.098098,0.239851
qwen3_vl_4b_i_sft_lora_ocr_conversion,880.0,1.0,0.594318,0.022727,0.022727,0.022727,0.233590,0.287489


### Split by other dimension

In [72]:
# conversion_df.groupby("_parse_status").apply(aggregate_func).to_csv(conversion_dir / "metrics_by_parse_status.csv")
conversion_df[conversion_df._parse_status == "success"].groupby(["model_name"]).apply(aggregate_func)

,num_samples,gt_valid_ratio,pred_valid_ratio,em_ratio,can_smiles_match_ratio,inchikey_match_ratio,tanimoto_sim_mean,tanimoto_sim_std
model_name,,,,,,,,
qwen3_4b_i,177.0,1.0,1.0,0.000000,0.197740,0.197740,0.297283,0.395280
qwen3_4b_i_sft_lora_conversion,794.0,1.0,1.0,0.827456,0.827456,0.827456,0.891324,0.266994
qwen3_vl_4b_i,148.0,1.0,1.0,0.006757,0.418919,0.418919,0.537854,0.435770
qwen3_vl_4b_i_grpo_lora_from_sft_lora_ocr_conversion,706.0,1.0,1.0,0.643059,0.643059,0.643059,0.759634,0.364141
qwen3_vl_4b_i_sft_lora_conversion,782.0,1.0,1.0,0.792839,0.792839,0.792839,0.869251,0.285243
qwen3_vl_4b_i_sft_lora_ocr,223.0,1.0,1.0,0.000000,0.197309,0.197309,0.366366,0.366214
qwen3_vl_4b_i_sft_lora_ocr_conversion,798.0,1.0,1.0,0.802005,0.802005,0.802005,0.879175,0.276186


In [73]:
# conversion_df.groupby(["output_rep_type"]).apply(aggregate_func).to_csv(conversion_dir / "metrics_by_output_rep_type.csv")
conversion_df.groupby(["model_name", "output_rep_type"]).apply(aggregate_func)

num_samples  \
model_name                                         output_rep_type                
qwen3_4b_i                                         can_deepsmiles         215.0   
                                                   can_selfies            210.0   
                                                   can_smiles             212.0   
                                                   inchi                  238.0   
qwen3_4b_i_sft_lora_conversion                     can_deepsmiles         215.0   
                                                   can_selfies            210.0   
                                                   can_smiles             212.0   
                                                   inchi                  238.0   
qwen3_vl_4b_i                                      can_deepsmiles         215.0   
                                                   can_selfies            210.0   
                                                   can_smiles             212.0   
                                                   inchi                  238.0   
qwen3_vl_4b_i_grpo_lora_from_sft_lora_ocr_conve... can_deepsmiles         215.0   
                                                   can_selfies            210.0   
                                                   can_smiles             212.0   
                                                   inchi                  238.0   
qwen3_vl_4b_i_sft_lora_conversion                  can_deepsmiles         215.0   
                                                   can_selfies            210.0   
                                                   can_smiles             212.0   
                                                   inchi                  238.0   
qwen3_vl_4b_i_sft_lora_ocr                         can_deepsmiles         215.0   
                                                   can_selfies            210.0   
                                                   can_smiles             212.0   
                                                   inchi                  238.0   
qwen3_vl_4b_i_sft_lora_ocr_conversion              can_deepsmiles         215.0   
                                                   can_selfies            210.0   
                                                   can_smiles             212.0   
                                                   inchi                  238.0   

                                                                    gt_valid_ratio  \
model_name                                         output_rep_type                   
qwen3_4b_i                                         can_deepsmiles              1.0   
                                                   can_selfies                 1.0   
                                                   can_smiles                  1.0   
                                                   inchi                       1.0   
qwen3_4b_i_sft_lora_conversion                     can_deepsmiles              1.0   
                                                   can_selfies                 1.0   
                                                   can_smiles                  1.0   
                                                   inchi                       1.0   
qwen3_vl_4b_i                                      can_deepsmiles              1.0   
                                                   can_selfies                 1.0   
                                                   can_smiles                  1.0   
                                                   inchi                       1.0   
qwen3_vl_4b_i_grpo_lora_from_sft_lora_ocr_conve... can_deepsmiles              1.0   
                                                   can_selfies                 1.0   
                                                   can_smiles                  1.0   
                                                   inchi                       1.0   
qwen3_vl_4b_i_sft_lora_conver

In [74]:
conversion_df.groupby(["input_rep_type"]).apply(aggregate_func)

,num_samples,gt_valid_ratio,pred_valid_ratio,em_ratio,can_smiles_match_ratio,inchikey_match_ratio,tanimoto_sim_mean,tanimoto_sim_std
input_rep_type,,,,,,,,
can_deepsmiles,434.0,1.0,0.610599,0.453917,0.453917,0.453917,0.501586,0.478962
can_selfies,455.0,1.0,0.527473,0.459341,0.459341,0.459341,0.484213,0.491825
can_smiles,378.0,1.0,0.571429,0.462963,0.462963,0.462963,0.485287,0.486543
cml,532.0,1.0,0.543233,0.296992,0.296992,0.296992,0.374278,0.450894
inchi,462.0,1.0,0.508658,0.086580,0.086580,0.086580,0.197127,0.317626
iupac,483.0,1.0,0.523810,0.196687,0.196687,0.196687,0.280539,0.398734
random_deepsmiles,1148.0,1.0,0.615854,0.465157,0.479965,0.479965,0.516745,0.484873
random_selfies,1106.0,1.0,0.665461,0.403255,0.492767,0.492767,0.542460,0.473585
random_smiles,1127.0,1.0,0.609583,0.459627,0.480923,0.480923,0.531458,0.481564


In [75]:
# conversion_df.groupby(["task_type"]).apply(aggregate_func).to_csv(conversion_dir / "metrics_by_task_type.csv")
conversion_df.groupby(["model_name", "task_type"]).apply(aggregate_func)

num_samples  \
model_name                                         task_type                              
qwen3_4b_i                                         can_reps_translation           247.0   
                                                   cml_understanding               76.0   
                                                   cross_rep_normalization        369.0   
                                                   intra_rep_normalization        114.0   
                                                   iupac_understanding             69.0   
qwen3_4b_i_sft_lora_conversion                     can_reps_translation           247.0   
                                                   cml_understanding               76.0   
                                                   cross_rep_normalization        369.0   
                                                   intra_rep_normalization        114.0   
                                                   iupac_understanding             69.0   
qwen3_vl_4b_i                                      can_reps_translation           247.0   
                                                   cml_understanding               76.0   
                                                   cross_rep_normalization        369.0   
                                                   intra_rep_normalization        114.0   
                                                   iupac_understanding             69.0   
qwen3_vl_4b_i_grpo_lora_from_sft_lora_ocr_conve... can_reps_translation           247.0   
                                                   cml_understanding               76.0   
                                                   cross_rep_normalization        369.0   
                                                   intra_rep_normalization        114.0   
                                                   iupac_understanding             69.0   
qwen3_vl_4b_i_sft_lora_conversion                  can_reps_translation           247.0   
                                                   cml_understanding               76.0   
                                                   cross_rep_normalization        369.0   
                                                   intra_rep_normalization        114.0   
                                                   iupac_understanding             69.0   
qwen3_vl_4b_i_sft_lora_ocr                         can_reps_translation           247.0   
                                                   cml_understanding               76.0   
                                                   cross_rep_normalization        369.0   
                                                   intra_rep_normalization        114.0   
                                                   iupac_understanding             69.0   
qwen3_vl_4b_i_sft_lora_ocr_conversion              can_reps_translation           247.0   
                                                   cml_understanding               76.0   
                                                   cross_rep_normalization        369.0   
                                                   intra_rep_normalization        114.0   
                                                   iupac_understanding             69.0   

                                                                            gt_valid_ratio  \
model_name                                         task_type                                 
qwen3_4b_i                                         can_reps_translation                1.0   
                                                   cml_understanding                   1.0   
                                                   cross_rep_normalization             1.0   
                                                   intra_rep_normalization             1.0   
                                                   iupac_understanding                 1.0   
qwen3_4b_i_sft_lora_conversion                    

In [76]:
conversion_df.groupby(["model_name", "task_id"]).apply(aggregate_func)

num_samples  \
model_name                            task_id                                                          
qwen3_4b_i                            can_reps_translation:can_deepsmiles->can_selfies          19.0   
                                      can_reps_translation:can_deepsmiles->can_smiles           22.0   
                                      can_reps_translation:can_deepsmiles->inchi                21.0   
                                      can_reps_translation:can_selfies->can_deepsmiles          25.0   
                                      can_reps_translation:can_selfies->can_smiles              16.0   
...                                                                                              ...   
qwen3_vl_4b_i_sft_lora_ocr_conversion intra_rep_normalization:random_smiles->can_smiles         34.0   
                                      iupac_understanding:iupac->can_deepsmiles                 14.0   
                                      iupac_understanding:iupac->can_selfies                    22.0   
                                      iupac_understanding:iupac->can_smiles                     13.0   
                                      iupac_understanding:iupac->inchi                          20.0   

                                                                                         gt_valid_ratio  \
model_name                            task_id                                                             
qwen3_4b_i                            can_reps_translation:can_deepsmiles->can_selfies              1.0   
                                      can_reps_translation:can_deepsmiles->can_smiles               1.0   
                                      can_reps_translation:can_deepsmiles->inchi                    1.0   
                                      can_reps_translation:can_selfies->can_deepsmiles              1.0   
                                      can_reps_translation:can_selfies->can_smiles                  1.0   
...                                                                                                 ...   
qwen3_vl_4b_i_sft_lora_ocr_conversion intra_rep_normalization:random_smiles->can_smiles             1.0   
                                      iupac_understanding:iupac->can_deepsmiles                     1.0   
                                      iupac_understanding:iupac->can_selfies                        1.0   
                                      iupac_understanding:iupac->can_smiles                         1.0   
                                      iupac_understanding:iupac->inchi                              1.0   

                                                                                         pred_valid_ratio  \
model_name                            task_id                                                               
qwen3_4b_i                            can_reps_translation:can_deepsmiles->can_selfies           0.315789   
                                      can_reps_translation:can_deepsmiles->can_smiles            0.227273   
                                      can_reps_translation:can_deepsmiles->inchi                 0.000000   
                                      can_reps_translation:can_selfies->can_deepsmiles           0.080000   
                                      can_reps_translation:can_selfies->can_smiles               0.187500   
...                                                                                                   ...   
qwen3_vl_4b_i_sft_lora_ocr_conversion intra_rep_normalization:random_smiles->can_smiles          1.000000   
                                      iupac_understanding:iupac->can_deepsmiles                  0.642857   
                                      iupac_understanding:iupac->can_selfies                     1.000000   
                                      iupac_understanding:iupac->can_smiles                      0.923077   
                                    